# 02 Rolling Features

Build rolling 7-day, 14-day, and season-to-date hitter features.



In [43]:
import pandas as pd
import numpy as np


In [44]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.data_loader import save_csv

print(PROJECT_ROOT)

/Users/henry_tsai/Desktop/daily-mlb-hitter-forecasting


In [45]:
pa_path = PROJECT_ROOT / "data/processed/pa_2025_04_01_to_05_31.csv"

pa_df = pd.read_csv(pa_path)
pa_df["game_date"] = pd.to_datetime(pa_df["game_date"])

print(pa_df.shape)
print(pa_df["game_date"].min(), pa_df["game_date"].max())
pa_df.head()

(60470, 13)
2025-04-01 00:00:00 2025-05-31 00:00:00


,game_date,batter,pitcher,events,description,pitch_type,stand,p_throws,launch_speed,launch_angle,estimated_woba_using_speedangle,woba_value,xwoba_value
0,2025-05-31,691785,606160,field_out,hit_into_play,FF,L,R,91.3,33.0,0.057000,0.0,0.057000
1,2025-05-31,665966,606160,field_out,hit_into_play,SL,R,R,71.4,-20.0,0.036000,0.0,0.036000
2,2025-05-31,677800,606160,field_out,hit_into_play,FF,L,R,103.4,36.0,0.975773,0.0,0.975773
3,2025-05-31,663897,595897,strikeout,swinging_strike,SL,R,R,NaN,NaN,0.000000,0.0,0.000000
4,2025-05-31,671739,595897,field_out,hit_into_play,SL,L,R,100.8,4.0,0.542000,0.0,0.542000


In [46]:
print(pa_df["game_date"].min())
print(pa_df["game_date"].max())
print(pa_df["game_date"].nunique())

2025-04-01 00:00:00
2025-05-31 00:00:00
61


In [47]:
needed_cols = [
    "game_date",
    "batter",
    "player_name",
    "events",
    "xwoba_value",
    "woba_value",
    "launch_speed",
    "launch_angle",
]

missing_cols = [c for c in needed_cols if c not in pa_df.columns]
print("Missing columns:", missing_cols)

Missing columns: ['player_name']


In [48]:
daily_hitter = (
    pa_df
    .groupby(["batter", "game_date"])
    .agg(
        PA=("events", "count"),
        xwOBA=("xwoba_value", "mean"),
        wOBA=("woba_value", "mean"),
        avg_exit_velocity=("launch_speed", "mean"),
        avg_launch_angle=("launch_angle", "mean"),
    )
    .reset_index()
    .sort_values(["batter", "game_date"])
)

print(daily_hitter.shape)
daily_hitter.head(20)

(15800, 7)


,batter,game_date,PA,xwOBA,wOBA,avg_exit_velocity,avg_launch_angle
0,455117,2025-04-02,4,0.046000,0.000000,72.766667,18.333333
1,455117,2025-04-04,2,0.711606,1.000000,96.250000,14.000000
2,455117,2025-04-08,4,0.539750,0.675000,89.275000,12.250000
3,455117,2025-04-09,4,0.029750,0.225000,93.100000,-12.000000
4,455117,2025-04-12,3,0.498980,0.000000,65.366667,-10.333333
5,455117,2025-04-14,2,0.056500,0.000000,24.900000,45.000000
6,455117,2025-04-15,3,0.230499,0.233333,NaN,NaN
7,455117,2025-04-18,4,0.079750,0.000000,90.833333,20.000000
8,455117,2025-04-20,4,0.650989,0.450000,94.850000,13.750000
9,455117,2025-04-21,3,0.277333,0.000000,91.300000,4.500000


In [49]:
def build_rolling_features_for_sample(
    daily_df,
    past_windows=(1, 2),
    target_window=1,
    min_pa_past=1,
    min_pa_future=1,
):
    """
    Build rolling hitter features from daily hitter data.

    Each row = one hitter on one prediction date.

    For each prediction date t:
    - past features use dates [t - window, t - 1]
    - target uses dates [t + 1, t + target_window]

    This avoids using same-day or future information in features.
    """

    daily_df = daily_df.copy()
    daily_df["game_date"] = pd.to_datetime(daily_df["game_date"])

    all_rows = []

    # Use all available dates as possible prediction dates
    all_dates = sorted(daily_df["game_date"].unique())

    for pred_date in all_dates:
        for batter, batter_df in daily_df.groupby("batter"):
            batter_df = batter_df.sort_values("game_date")

            row = {
                "batter": batter,
                "prediction_date": pred_date,
            }

            # player_name
            # names = batter_df["player_name"].dropna().unique()
            # row["player_name"] = names[0] if len(names) > 0 else None

            # Past rolling features
            valid_past = True

            for window in past_windows:
                start_date = pred_date - pd.Timedelta(days=window)
                end_date = pred_date - pd.Timedelta(days=1)

                past = batter_df[
                    (batter_df["game_date"] >= start_date)
                    & (batter_df["game_date"] <= end_date)
                ]

                pa_sum = past["PA"].sum()

                row[f"last_{window}d_PA"] = pa_sum

                if pa_sum >= min_pa_past:
                    row[f"last_{window}d_xwOBA"] = np.average(
                        past["xwOBA"],
                        weights=past["PA"]
                    )
                    row[f"last_{window}d_wOBA"] = np.average(
                        past["wOBA"],
                        weights=past["PA"]
                    )
                    row[f"last_{window}d_avg_exit_velocity"] = np.average(
                        past["avg_exit_velocity"].dropna()
                    ) if past["avg_exit_velocity"].notna().any() else np.nan
                    row[f"last_{window}d_avg_launch_angle"] = np.average(
                        past["avg_launch_angle"].dropna()
                    ) if past["avg_launch_angle"].notna().any() else np.nan
                else:
                    row[f"last_{window}d_xwOBA"] = np.nan
                    row[f"last_{window}d_wOBA"] = np.nan
                    row[f"last_{window}d_avg_exit_velocity"] = np.nan
                    row[f"last_{window}d_avg_launch_angle"] = np.nan

            # Season-to-date features:
            # use all dates before prediction_date
            season = batter_df[batter_df["game_date"] < pred_date]
            season_pa = season["PA"].sum()

            row["season_PA"] = season_pa

            if season_pa >= min_pa_past:
                row["season_xwOBA"] = np.average(season["xwOBA"], weights=season["PA"])
                row["season_wOBA"] = np.average(season["wOBA"], weights=season["PA"])
                row["season_avg_exit_velocity"] = (
                    season["avg_exit_velocity"].dropna().mean()
                    if season["avg_exit_velocity"].notna().any()
                    else np.nan
                )
                row["season_avg_launch_angle"] = (
                    season["avg_launch_angle"].dropna().mean()
                    if season["avg_launch_angle"].notna().any()
                    else np.nan
                )
            else:
                row["season_xwOBA"] = np.nan
                row["season_wOBA"] = np.nan
                row["season_avg_exit_velocity"] = np.nan
                row["season_avg_launch_angle"] = np.nan

            # Future target:
            # use dates [prediction_date + 1, prediction_date + target_window]
            future_start = pred_date + pd.Timedelta(days=1)
            future_end = pred_date + pd.Timedelta(days=target_window)

            future = batter_df[
                (batter_df["game_date"] >= future_start)
                & (batter_df["game_date"] <= future_end)
            ]

            future_pa = future["PA"].sum()
            row[f"future_{target_window}d_PA"] = future_pa

            if future_pa >= min_pa_future:
                row[f"future_{target_window}d_xwOBA"] = np.average(
                    future["xwOBA"],
                    weights=future["PA"]
                )
            else:
                row[f"future_{target_window}d_xwOBA"] = np.nan

            all_rows.append(row)

    feature_df = pd.DataFrame(all_rows)

    # Add trend features
    for window in past_windows:
        feature_df[f"xwOBA_diff_{window}d_vs_season"] = (
            feature_df[f"last_{window}d_xwOBA"] - feature_df["season_xwOBA"]
        )

        feature_df[f"wOBA_diff_{window}d_vs_season"] = (
            feature_df[f"last_{window}d_wOBA"] - feature_df["season_wOBA"]
        )

    return feature_df

In [51]:
feature_df = build_rolling_features_for_sample(
    daily_hitter,
    past_windows=(7, 14),
    target_window=7,
    min_pa_past=5,
    min_pa_future=5,
)

print(feature_df.shape)
feature_df.head()

(32635, 23)


,batter,prediction_date,last_7d_PA,last_7d_xwOBA,last_7d_wOBA,last_7d_avg_exit_velocity,last_7d_avg_launch_angle,last_14d_PA,last_14d_xwOBA,last_14d_wOBA,...,season_xwOBA,season_wOBA,season_avg_exit_velocity,season_avg_launch_angle,future_7d_PA,future_7d_xwOBA,xwOBA_diff_7d_vs_season,wOBA_diff_7d_vs_season,xwOBA_diff_14d_vs_season,wOBA_diff_14d_vs_season
0,455117,2025-04-01,0,NaN,NaN,NaN,NaN,0,NaN,NaN,...,NaN,NaN,NaN,NaN,10,0.376621,NaN,NaN,NaN,NaN
1,456781,2025-04-01,0,NaN,NaN,NaN,NaN,0,NaN,NaN,...,NaN,NaN,NaN,NaN,9,0.245931,NaN,NaN,NaN,NaN
2,457705,2025-04-01,0,NaN,NaN,NaN,NaN,0,NaN,NaN,...,NaN,NaN,NaN,NaN,13,0.347961,NaN,NaN,NaN,NaN
3,457759,2025-04-01,0,NaN,NaN,NaN,NaN,0,NaN,NaN,...,NaN,NaN,NaN,NaN,16,0.413958,NaN,NaN,NaN,NaN
4,467793,2025-04-01,0,NaN,NaN,NaN,NaN,0,NaN,NaN,...,NaN,NaN,NaN,NaN,21,0.372271,NaN,NaN,NaN,NaN


In [52]:
model_ready = feature_df.dropna(
    subset=[
        "last_7d_xwOBA",
        "last_14d_xwOBA",
        "season_xwOBA",
        "future_7d_xwOBA",
    ]
).copy()

print(model_ready.shape)
model_ready.head()

(18350, 23)


,batter,prediction_date,last_7d_PA,last_7d_xwOBA,last_7d_wOBA,last_7d_avg_exit_velocity,last_7d_avg_launch_angle,last_14d_PA,last_14d_xwOBA,last_14d_wOBA,...,season_xwOBA,season_wOBA,season_avg_exit_velocity,season_avg_launch_angle,future_7d_PA,future_7d_xwOBA,xwOBA_diff_7d_vs_season,wOBA_diff_7d_vs_season,xwOBA_diff_14d_vs_season,wOBA_diff_14d_vs_season
546,518595,2025-04-02,5,0.243464,0.18,92.080000,23.400000,5,0.243464,0.18,...,0.243464,0.18,92.080000,23.400000,6,0.263810,0.0,0.0,0.0,0.0
560,545361,2025-04-02,5,0.427303,0.33,89.500000,27.333333,5,0.427303,0.33,...,0.427303,0.33,89.500000,27.333333,24,0.420977,0.0,0.0,0.0,0.0
566,571448,2025-04-02,5,0.103000,0.36,95.075000,-1.250000,5,0.103000,0.36,...,0.103000,0.36,95.075000,-1.250000,22,0.241077,0.0,0.0,0.0,0.0
625,621043,2025-04-02,5,0.103800,0.00,97.033333,3.000000,5,0.103800,0.00,...,0.103800,0.00,97.033333,3.000000,25,0.304311,0.0,0.0,0.0,0.0
627,621439,2025-04-02,5,0.343233,0.32,92.500000,35.000000,5,0.343233,0.32,...,0.343233,0.32,92.500000,35.000000,21,0.187231,0.0,0.0,0.0,0.0


In [53]:
model_ready["future_7d_xwOBA"].describe()

count    18350.000000
mean         0.313295
std          0.099286
min          0.000000
25%          0.248339
50%          0.309385
75%          0.373892
max          0.874359
Name: future_7d_xwOBA, dtype: float64

In [54]:
print(model_ready["prediction_date"].min())
print(model_ready["prediction_date"].max())

2025-04-02 00:00:00
2025-05-30 00:00:00


In [55]:
daily_hitter.groupby(["batter", "game_date"]).size().sort_values(ascending=False).head()

batter  game_date 
455117  2025-04-02    1
672515  2025-04-08    1
        2025-04-12    1
        2025-04-15    1
        2025-04-16    1
dtype: int64

In [56]:
daily_hitter[daily_hitter["batter"] == batter_id].sort_values("game_date")

,batter,game_date,PA,xwOBA,wOBA,avg_exit_velocity,avg_launch_angle
13133,682998,2025-04-01,5,0.332921,0.400000,87.050000,-5.500000
13134,682998,2025-04-02,5,0.444530,0.140000,94.650000,46.000000
13135,682998,2025-04-03,5,0.216600,0.180000,90.225000,6.000000
13136,682998,2025-04-04,5,0.917041,0.940000,103.700000,37.333333
13137,682998,2025-04-05,4,0.405874,0.175000,96.266667,25.000000
13138,682998,2025-04-06,5,0.399000,0.750000,97.625000,4.500000
13139,682998,2025-04-07,4,0.443000,0.312500,91.450000,-1.250000
13140,682998,2025-04-08,4,0.619263,0.900000,103.750000,12.000000
13141,682998,2025-04-09,5,0.686224,0.720000,104.233333,15.000000
13142,682998,2025-04-11,4,0.179000,0.225000,96.400000,11.666667


In [57]:
top_batter = daily_hitter.groupby("batter")["PA"].sum().reset_index()
top_batter = top_batter.sort_values("PA", ascending=False)

top_batter.head(10)

,batter,PA
404,680776,254
146,646240,253
56,596019,251
438,682998,249
75,607208,248
45,592450,243
498,694192,243
255,666182,242
509,695578,242
188,663457,240


In [58]:
batter_id = top_batter.iloc[0]["batter"]

daily_hitter[daily_hitter["batter"] == batter_id].sort_values("game_date")

,batter,game_date,PA,xwOBA,wOBA,avg_exit_velocity,avg_launch_angle
12212,680776,2025-04-02,5,0.181200,0.000000,100.900000,-2.333333
12213,680776,2025-04-03,5,0.305676,0.000000,96.700000,22.000000
12214,680776,2025-04-04,6,0.504469,0.300000,93.366667,18.833333
12215,680776,2025-04-06,10,0.106200,0.270000,91.285714,-3.571429
12216,680776,2025-04-07,4,0.437923,0.800000,96.533333,14.666667
12217,680776,2025-04-08,4,0.172874,0.175000,NaN,NaN
12218,680776,2025-04-09,5,0.229000,0.180000,86.525000,-8.000000
12219,680776,2025-04-10,5,0.508999,0.640000,96.300000,11.000000
12220,680776,2025-04-11,5,0.116800,0.000000,79.000000,-5.333333
12221,680776,2025-04-13,5,0.132800,0.180000,97.833333,10.000000


In [59]:
feature_sample[feature_sample["batter"] == batter_id].sort_values("prediction_date")

,batter,prediction_date,last_1d_PA,last_1d_xwOBA,last_1d_wOBA,last_1d_avg_exit_velocity,last_1d_avg_launch_angle,last_2d_PA,last_2d_xwOBA,last_2d_wOBA,...,season_xwOBA,season_wOBA,season_avg_exit_velocity,season_avg_launch_angle,future_1d_PA,future_1d_xwOBA,xwOBA_diff_1d_vs_season,wOBA_diff_1d_vs_season,xwOBA_diff_2d_vs_season,wOBA_diff_2d_vs_season
294,680776,2025-04-01,0,NaN,NaN,NaN,NaN,0,NaN,NaN,...,NaN,NaN,NaN,NaN,5,0.181200,NaN,NaN,NaN,NaN
658,680776,2025-04-02,0,NaN,NaN,NaN,NaN,0,NaN,NaN,...,NaN,NaN,NaN,NaN,5,0.305676,NaN,NaN,NaN,NaN
1022,680776,2025-04-03,5,0.1812,0.0,100.9,-2.333333,5,0.1812,0.0,...,0.1812,0.0,100.9,-2.333333,0,NaN,0.0,0.0,0.0,0.0


In [60]:
feature_path = PROJECT_ROOT / "data/features/rolling_features_2025_04_01_to_05_31.csv"
save_csv(feature_df, feature_path)

model_ready_path = PROJECT_ROOT / "data/features/model_ready_2025_04_01_to_05_31.csv"
save_csv(model_ready, model_ready_path)

print(feature_path.exists())
print(model_ready_path.exists())

True
True


## Summary

This notebook converts PA-level Statcast data into hitter-level rolling features.

For the sample dataset, we use short test windows because only three days of data are available:

- Past windows: 1 day and 2 days
- Target window: future 1 day

The final output is one row per hitter per prediction date, including recent-form features, season-to-date features, trend/deviation features, and a future xwOBA target.

This confirms that the rolling-window forecasting pipeline works before scaling to full 7-day, 14-day, and future 7-day windows.